# RAG Exploration Notebook

This notebook explores the committed sample documents and the retrieval stack implemented in this project. It is intended for portfolio review and local experimentation, not for production serving.

Default cells avoid paid API calls. Cells that may call an LLM or a slow model are gated behind explicit flags.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd
from langchain_community.vectorstores import FAISS

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "project-1-rag-qa-plan.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from eval.rag_eval import THRESHOLDS, load_eval_dataset  # noqa: E402
from src.embedder import get_embedding_model  # noqa: E402
from src.ingest import chunk_documents, load_documents  # noqa: E402
from src.retriever import HybridRetriever  # noqa: E402

DOCS_PATH = PROJECT_ROOT / "data" / "sample_docs"
EVAL_DATASET_PATH = PROJECT_ROOT / "data" / "eval_dataset.json"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

pd.set_option("display.max_colwidth", 120)

## 1. Document Loading And Metadata

Load the committed sample PDFs and inspect the ingestion metadata added by `src.ingest.load_documents`.

In [ ]:
documents = load_documents(DOCS_PATH)

metadata_df = pd.DataFrame(
    [
        {
            "source_file": doc.metadata.get("source_file"),
            "file_type": doc.metadata.get("file_type"),
            "file_size_bytes": doc.metadata.get("file_size_bytes"),
            "page": doc.metadata.get("page"),
            "characters": len(doc.page_content),
            "ingested_at": doc.metadata.get("ingested_at"),
        }
        for doc in documents
    ]
)

metadata_df.head(10)

In [ ]:
metadata_df.groupby("source_file").agg(
    pages=("page", "count"),
    total_characters=("characters", "sum"),
    file_size_bytes=("file_size_bytes", "max"),
).sort_values("source_file")

## 2. Chunk Size Experiment

Compare chunk length distributions for `chunk_size` values of 200, 500, and 1000. The production default is 500 with overlap 50.

In [ ]:
chunk_experiments: list[dict[str, int | str]] = []

for chunk_size in [200, 500, 1000]:
    chunks_for_size = chunk_documents(
        documents, chunk_size=chunk_size, chunk_overlap=50
    )
    for chunk in chunks_for_size:
        chunk_experiments.append(
            {
                "chunk_size": chunk_size,
                "source_file": chunk.metadata.get("source_file", "unknown"),
                "length": len(chunk.page_content),
            }
        )

chunk_lengths_df = pd.DataFrame(chunk_experiments)
chunk_lengths_df.groupby("chunk_size")["length"].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, chunk_size in zip(axes, [200, 500, 1000]):
    lengths = chunk_lengths_df.loc[
        chunk_lengths_df["chunk_size"] == chunk_size, "length"
    ]
    ax.hist(lengths, bins=20, color="#2f6f73", edgecolor="white")
    ax.set_title(f"chunk_size={chunk_size}")
    ax.set_xlabel("Characters")

axes[0].set_ylabel("Chunk count")
fig.suptitle("Chunk Length Distribution By Configured Size")
fig.tight_layout()

## 3. Embedding Space UMAP

Embed the first 50 chunks and project them into two dimensions with UMAP, colored by source document.

In [ ]:
chunks = chunk_documents(documents, chunk_size=500, chunk_overlap=50)
sample_chunks = chunks[:50]

embedding_model = get_embedding_model(EMBEDDING_MODEL)
embeddings = embedding_model.embed_documents(
    [chunk.page_content for chunk in sample_chunks]
)

len(sample_chunks), len(embeddings), len(embeddings[0])

In [ ]:
import umap

projection = umap.UMAP(n_components=2, random_state=42, metric="cosine").fit_transform(
    embeddings
)
umap_df = pd.DataFrame(
    {
        "x": projection[:, 0],
        "y": projection[:, 1],
        "source_file": [
            chunk.metadata.get("source_file", "unknown") for chunk in sample_chunks
        ],
        "chunk_index": [chunk.metadata.get("chunk_index") for chunk in sample_chunks],
    }
)

fig, ax = plt.subplots(figsize=(8, 6))
for source_file, group in umap_df.groupby("source_file"):
    ax.scatter(group["x"], group["y"], label=source_file, alpha=0.75, s=45)

ax.set_title("UMAP Projection Of 50 Embedded Chunks")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.legend(loc="best", fontsize=8)
fig.tight_layout()

## 4. Dense Vs BM25 Vs Hybrid Retrieval

Build an in-memory FAISS index and compare retrieval modes for five representative queries.

In [ ]:
vectorstore = FAISS.from_documents(chunks, embedding_model)
retrieval_settings = SimpleNamespace(fetch_k_multiplier=4, mmr_lambda=0.7)
retriever = HybridRetriever(vectorstore, chunks, retrieval_settings)

queries = [
    "What are the four AI RMF core functions?",
    "How does zero trust handle implicit trust?",
    "What risks are specific to generative AI?",
    "Why does policy matter in zero trust architecture?",
    "What makes an AI system trustworthy?",
]


def summarize_docs(docs):
    return [
        f"{doc.metadata.get('source_file')}#chunk-{doc.metadata.get('chunk_index')}: "
        f"{doc.page_content[:120].replace(chr(10), ' ')}"
        for doc in docs
    ]


comparison_rows = []
for query in queries:
    dense_docs = [doc for doc, _ in retriever.retrieve_dense(query, k=3, fetch_k=12)]
    bm25_docs = [doc for doc, _ in retriever.retrieve_bm25(query, k=3)]
    hybrid_docs = retriever.retrieve_hybrid(query, k=3)
    comparison_rows.append(
        {
            "query": query,
            "dense_top3": summarize_docs(dense_docs),
            "bm25_top3": summarize_docs(bm25_docs),
            "hybrid_top3": summarize_docs(hybrid_docs),
        }
    )

pd.DataFrame(comparison_rows)

## 5. Re-Ranking Impact

The CrossEncoder is a heavier model than the bi-encoder retriever, so this cell is opt-in. Set `RUN_RERANKER = True` to compare before/after ordering for a test query.

In [ ]:
RUN_RERANKER = False
test_query = "What does zero trust say about implicit trust?"
candidates = retriever.retrieve_hybrid(test_query, k=8)

if RUN_RERANKER:
    from src.reranker import CrossEncoderReranker

    reranker = CrossEncoderReranker("cross-encoder/ms-marco-MiniLM-L-6-v2")
    reranked = reranker.rerank(test_query, candidates, top_n=5)
else:
    reranked = candidates[:5]

pd.DataFrame(
    {
        "rank_before": range(1, len(candidates[:5]) + 1),
        "before": summarize_docs(candidates[:5]),
        "rank_after": range(1, len(reranked) + 1),
        "after": summarize_docs(reranked),
        "reranker_enabled": RUN_RERANKER,
    }
)

## 6. HyDE Comparison

HyDE requires an LLM call in the full app. This notebook defaults to a deterministic stand-in expansion so the retrieval comparison can be reviewed without secrets or paid calls.

In [ ]:
hyde_query = "How should organizations manage generative AI risk?"
deterministic_hyde_expansion = (
    "Organizations manage generative AI risk through governance, mapping risks, "
    "measuring impacts, managing harmful outputs, monitoring information integrity, "
    "and applying the NIST AI Risk Management Framework profile for generative AI."
)

raw_docs = retriever.retrieve_hybrid(hyde_query, k=5)
hyde_docs = retriever.retrieve_hybrid(deterministic_hyde_expansion, k=5)

pd.DataFrame(
    {
        "rank": range(1, 6),
        "raw_query_results": summarize_docs(raw_docs),
        "hyde_expanded_results": summarize_docs(hyde_docs),
    }
)

## 7. RAGAS Eval Score Table

The eval dataset has 10 questions. Running true RAGAS evaluation requires a configured pipeline and LLM credentials, so this notebook shows the dataset and threshold table by default. Set `RUN_RAGAS = True` only in a configured local environment.

In [ ]:
eval_dataset = load_eval_dataset(EVAL_DATASET_PATH)
pd.DataFrame(eval_dataset)[["question", "ground_truth"]].head(10)

In [ ]:
RUN_RAGAS = False

if RUN_RAGAS:
    from eval.rag_eval import run_ragas_evaluation
    from src.config import settings
    from src.pipeline import RAGPipeline

    pipeline = RAGPipeline(settings)
    scores = run_ragas_evaluation(pipeline, eval_dataset)
else:
    scores = {metric: None for metric in THRESHOLDS}

pd.DataFrame(
    [
        {
            "metric": metric,
            "score": scores.get(metric),
            "threshold": threshold,
            "status": "not run"
            if scores.get(metric) is None
            else ("pass" if scores[metric] >= threshold else "fail"),
        }
        for metric, threshold in THRESHOLDS.items()
    ]
)